In [ ]:
import os, re, warnings

import pandas as pd

warnings.filterwarnings("ignore")

In [ ]:
ELECTIONS = {
    "US_2024": {
        "since"        : "2024-07-05T00:00:00Z",
        "until"        : "2024-11-04T23:59:59Z",
        "election_date": "2024-11-05",
        "hashtags": [
            # ── Core election ──────────────────────────────────────────────
            "#Election2024", "#USElection2024", "#ElectionDay", "#Vote2024",
            "#Presidential2024", "#PresidentialElection", "#ElectionNight",
            "#AmericaDecides", "#Decision2024", "#VoterRegistration",
            "#EarlyVoting", "#MailInVoting", "#ElectionIntegrity",
            "#Battleground2024", "#SwingState",

            # ── Trump / Republican ─────────────────────────────────────────
            "#Trump2024", "#TrumpVance", "#VoteTrump", "#Trump",
            "#DonaldTrump", "#MAGA", "#MAGA2024", "#AmericaFirst",
            "#TrumpRally", "#Republicans", "#GOP", "#Republican",
            "#JDVance", "#Vance2024", "#VanceVP",
            "#RNC2024", "#RepublicanConvention",
            "#Project2025", "#TrumpDebate",

            # ── Harris / Democrat ──────────────────────────────────────────
            "#Harris2024", "#KamalaHarris2024", "#KamalaHarris",
            "#HarrisWalz", "#VoteHarris", "#VoteBlue", "#VoteKamala",
            "#Kamala2024", "#Kamala", "#Harris",
            "#TimWalz", "#Walz2024", "#WalzVP",
            "#DNC2024", "#DemConvention", "#DemocraticConvention",
            "#WeAreNotGoingBack", "#WinWithKamala",
            "#Democrats", "#Democrat",

            # ── Biden exit ─────────────────────────────────────────────────
            "#BidenDropsOut", "#BidenOut",
            "#BidenWithdraws",

            # ── Debates & key moments ──────────────────────────────────────
            "#PresidentialDebate", "#Debate2024", "#VPDebate",
            "#TrumpHarrisDebate", "#DebateNight",
        ],
    },
}

ACTIVE_ELECTION = "US_2024"
cfg             = ELECTIONS[ACTIVE_ELECTION]
MIN_TEXT_LENGTH = 30

print(f"Active election : {ACTIVE_ELECTION}")
print(f"Window          : {cfg['since'][:10]}  to  {cfg['until'][:10]}")
print(f"Hashtags        : {len(cfg['hashtags'])}")

In [ ]:
# ── Text cleaning ─────────────────────────────────────────────────────────────
def clean_text(text):
    """Strip URLs and collapse whitespace."""
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ── Quality filters ────────────────────────────────────────────────────────────
NEWS_BOT_PATTERNS = [
    "forbes", "guardian", "bbc", "cnn", "reuters", "apnews",
    "skynews", "france24", "euronews", "washingtonpost", "nytimes",
    "independent", "trtworld", "straits_times", "abc", "news-feed",
    "theguardian", "huffpost", "politico", "axios",
]

def is_news_bot(handle):
    return any(p in handle.lower() for p in NEWS_BOT_PATTERNS)

def has_real_text(text):
    """Return True if text has enough non-hashtag/mention content."""
    clean = re.sub(r"#\w+|@\w+", "", text).strip()
    return len(clean) >= MIN_TEXT_LENGTH


# ── Buzz labelling — query-based ───────────────────────────────────────────────
# Labels indicate which hashtag cluster a post belongs to, NOT the author's stance.
# A post labelled "TrumpBuzz" was found via a Trump-related hashtag.
# It may be pro-Trump, anti-Trump, or simply reporting on Trump.

TRUMP_QUERIES = {q.lower() for q in [
    "#Trump2024", "#TrumpVance", "#VoteTrump", "#Trump", "#DonaldTrump",
    "#MAGA", "#MAGA2024", "#AmericaFirst", "#TrumpRally", "#Republicans",
    "#GOP", "#Republican", "#JDVance", "#Vance2024", "#VanceVP",
    "#RNC2024", "#RepublicanConvention", "#Project2025", "#TrumpDebate",
]}
HARRIS_QUERIES = {q.lower() for q in [
    "#Harris2024", "#KamalaHarris2024", "#KamalaHarris", "#HarrisWalz",
    "#VoteHarris", "#VoteBlue", "#VoteKamala", "#Kamala2024", "#Kamala",
    "#Harris", "#TimWalz", "#Walz2024", "#WalzVP", "#DNC2024",
    "#DemConvention", "#DemocraticConvention", "#WeAreNotGoingBack",
    "#WinWithKamala", "#Democrats", "#Democrat",
]}

def label_buzz(row):
    """
    Assign buzz label based on the hashtag query that retrieved the post.
    For comments (query='thread'), fall back to text-based keyword matching
    as a rough approximation — treat with caution.
    """
    query = str(row.get("query", "")).lower().strip()
    if query in TRUMP_QUERIES:
        return "TrumpBuzz"
    if query in HARRIS_QUERIES:
        return "HarrisBuzz"
    # Fallback for comments / neutral hashtags: keyword heuristic
    text = str(row.get("text", "")).lower()
    trump_kw  = ["trump", "maga", "donald", "republican", "gop", "vance"]
    harris_kw = ["harris", "kamala", "democrat", "democratic", "walz"]
    s_t = sum(k in text for k in trump_kw)
    s_h = sum(k in text for k in harris_kw)
    if s_t > s_h: return "TrumpBuzz"
    if s_h > s_t: return "HarrisBuzz"
    return "ElectionBuzz"

print("Helper functions ready: clean_text, is_news_bot, has_real_text, label_buzz")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BRONZE_CSV = f"../../Data/1_Bronze/Bluesky/bsky_{ACTIVE_ELECTION}_raw.csv"
SILVER_CSV = f"../../Data/2_Silver/Bluesky/bsky_{ACTIVE_ELECTION}_posts.csv"

os.makedirs(os.path.dirname(os.path.abspath(BRONZE_CSV)), exist_ok=True)
os.makedirs(os.path.dirname(os.path.abspath(SILVER_CSV)), exist_ok=True)

In [ ]:
# ── One-time fix: remove 'election' column from existing Bronze CSV ────────────
if os.path.exists(BRONZE_CSV):
    df_fix = pd.read_csv(BRONZE_CSV)
    if "election" in df_fix.columns:
        df_fix = df_fix.drop(columns=["election"])
        df_fix.to_csv(BRONZE_CSV, index=False)
        print(f"'election' column removed — {len(df_fix):,} rows saved → {BRONZE_CSV}")
    else:
        print("'election' column not present, nothing to do.")
else:
    print("Bronze CSV not found.")

In [ ]:
# ── Cleaning: filter, label and save to Silver ────────────────────────────────
# NOTE: the 'candidate' column reflects *hashtag cluster* (buzz), not political stance.
#   TrumpBuzz   = found via a Trump-related hashtag  (may be pro- OR anti-Trump)
#   HarrisBuzz  = found via a Harris-related hashtag (may be pro- OR anti-Harris)
#   ElectionBuzz = found via a neutral election hashtag, or no clear match

if not os.path.exists(BRONZE_CSV):
    print("Bronze CSV not found — run the scraping cell first.")
else:
    df     = pd.read_csv(BRONZE_CSV)
    before = len(df)

    # Parse timestamps
    df["datetime"] = pd.to_datetime(df["timestamp"], format="ISO8601", utc=True)

    # Keep only posts within the election window
    df = df[
        (df["datetime"] >= cfg["since"]) &
        (df["datetime"] <= cfg["until"])
    ]

    # Remove news bots and posts with too little real text
    df = df[~df["author"].apply(is_news_bot)]
    df = df[df["text"].apply(has_real_text)]

    # Buzz label: which hashtag cluster was this post found in?
    df["candidate"] = df.apply(label_buzz, axis=1)

    # Sort chronologically
    df = df.sort_values("datetime").reset_index(drop=True)

    # Column order
    cols = ["timestamp", "author", "display", "text", "candidate", "post_type",
            "likes", "reposts", "replies", "mentions", "is_reply", "parent_uri",
            "query", "uri"]
    cols = [c for c in cols if c in df.columns]
    df   = df[cols]

    df.to_csv(SILVER_CSV, index=False)

    print(f"Filtered  : {before:,} → {len(df):,} rows  ({before - len(df):,} removed)")
    print(f"Date range: {df.timestamp.iloc[0][:10]} → {df.timestamp.iloc[-1][:10]}")
    print(f"Buzz labels: {df.candidate.value_counts().to_dict()}")
    print(f"Silver saved → {SILVER_CSV}")